In [1]:
import pandas as pd
import re
df = pd.read_csv("ColetaSPIRA_RECENTE-clean.csv")
print(f"Tamanho do dataset: {df.shape}")

Tamanho do dataset: (1164, 50)


In [2]:
def filtrar_testes(df):
  # Preenche valores NaN na coluna 'local_coleta' com string vazia para evitar erros
  # e então filtra as linhas que contêm 'TESTE' (case-insensitive)
  df_filtrado = df[~df['local_coleta'].fillna('').str.contains('TESTE', case=False, na=False)]
  df_filtrado = df_filtrado[~df_filtrado['local_coleta'].fillna('').str.contains('TESTJAQUELINE', case=False, na=False)]
  df_filtrado = df_filtrado[~df_filtrado['local_coleta'].fillna('').str.contains('MARCELO', case=False, na=False)]
  df_filtrado = df_filtrado[~df_filtrado['local_coleta'].fillna('').str.contains('TEST', case=False, na=False)]
  #df_filtrado = df_filtrado[~df_filtrado['local_coleta'].fillna('').str.contains('TESTE', case=False, na=False)]
  df_filtrado = df_filtrado[~df_filtrado['local_coleta'].fillna('').str.contains('HOME', case=False, na=False)]
  df_filtrado = df_filtrado[~df_filtrado['data_coleta'].str.contains('2026-03-06 10:02:51.509238', na=False)]
  df_filtrado = df_filtrado[~df_filtrado['data_coleta'].str.contains('2026-03-06 09:06:32.244884', na=False)]
  return df_filtrado

df_filtrado = filtrar_testes(df)
print(f"Tamanho do dataset filtrado: {df_filtrado.shape}")

Tamanho do dataset filtrado: (1088, 50)


In [3]:
var = df['nome_linha_estudo'].value_counts()

In [4]:
df_filtrado['split'] = None

In [5]:
def gerar_bin(valor_total):
  bin = []
  valor_atual = valor_total
  for i in range(5, 0, -1):
    adicionar = valor_atual//i
    bin.append(int(adicionar))
    valor_atual -= adicionar
  #bin.append(valor_total - sum(bin))
  assert sum(bin) == valor_total, "Soma dos valores não é igual ao valor total"
  return bin

dic = {}
for c, v in var.items():
  bin = gerar_bin(v)
  dic[c] = bin
print(dic)

{'Asma': [114, 114, 115, 115, 115], 'Controle': [56, 56, 56, 56, 57], 'Insuficiência Respiratória': [36, 36, 36, 36, 37], 'Tabagismo': [22, 22, 22, 22, 23], 'Parkinson': [3, 3, 4, 4, 4]}


In [6]:
import random

random.seed(42)

In [7]:
def sortear_bin(bin):
  chave = True
  while chave:
    sorteado = random.randint(0, 4)
    if bin[sorteado] > 0:
      chave = False
      bin[sorteado] -= 1
  return sorteado

for idx, row in df_filtrado.iterrows():
  nome_linha = row['nome_linha_estudo']
  split = sortear_bin(dic[nome_linha])
  df_filtrado.at[idx, 'split'] = split

In [8]:
df_filtrado['split']

,split
0,0
1,0
2,2
3,1
4,1
...,...
1157,4
1158,2
1159,2
1160,0


In [9]:
df_filtrado.to_csv("arquivo2.csv", index=False)